In [1]:
import pandas as pd

In [2]:
ehr = pd.read_csv('./Datasets/mock_ehr_early_disease_dataset.csv')

In [3]:
ehr.head()

,patient_id,age,gender,bmi,blood_pressure,cholesterol_level,glucose_level,heart_rate,oxygen_saturation,diagnosis,medications,disease_risk
0,P00001,69,Female,25.7,119.0,191.0,67.0,85,94.8,Normal,NaN,0
1,P00002,32,Female,22.3,131.0,253.0,109.0,80,96.8,Hyperlipidemia,Atorvastatin,1
2,P00003,89,Female,27.8,126.0,250.0,127.0,81,97.3,Diabetes,Insulin,1
3,P00004,78,Male,20.8,126.0,230.0,97.0,92,95.7,Normal,NaN,0
4,P00005,38,Male,27.2,150.0,219.0,59.0,92,96.1,Hypertension,Amlodipine,1


In [4]:
ehr.shape

(5000, 12)

In [5]:
if 'patient_id' in ehr.columns:
    ehr.drop(columns=['patient_id'], inplace=True)

In [6]:
ehr.isnull().sum()

age                     0
gender                  0
bmi                     0
blood_pressure          0
cholesterol_level       0
glucose_level           0
heart_rate              0
oxygen_saturation       0
diagnosis               0
medications          2167
disease_risk            0
dtype: int64

In [7]:
ehr['medications'] = ehr['medications'].fillna('None')

In [8]:
for col in ehr.columns:
    if ehr[col].dtype in ["float64", "int64"]:
        ehr[col].fillna(ehr[col].mean(), inplace=True)
    else:
        ehr[col].fillna(ehr[col].mode()[0], inplace=True)

C:\Users\ashar\AppData\Local\Temp\ipykernel_5620\424484950.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  ehr[col].fillna(ehr[col].mean(), inplace=True)
C:\Users\ashar\AppData\Local\Temp\ipykernel_5620\424484950.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, 

In [9]:
if "patient_id" in ehr.columns:
    ehr.drop(columns=["patient_id"], inplace=True)

In [10]:
ehr.columns

Index(['age', 'gender', 'bmi', 'blood_pressure', 'cholesterol_level',
       'glucose_level', 'heart_rate', 'oxygen_saturation', 'diagnosis',
       'medications', 'disease_risk'],
      dtype='object')

In [11]:
from sklearn.preprocessing import LabelEncoder

In [12]:
label_encoder = LabelEncoder()
ehr['diagnosis'] = label_encoder.fit_transform(ehr['diagnosis'])
ehr['diagnosis']

0       3
1       1
2       0
3       3
4       2
       ..
4995    3
4996    1
4997    4
4998    2
4999    0
Name: diagnosis, Length: 5000, dtype: int64

In [13]:
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("Diagnosis Label Encoding Mapping:")
print(label_mapping)

Diagnosis Label Encoding Mapping:
{'Diabetes': np.int64(0), 'Hyperlipidemia': np.int64(1), 'Hypertension': np.int64(2), 'Normal': np.int64(3), 'Obesity': np.int64(4)}


In [14]:
ehr["metabolic_pathway_score"] = (
    0.4 * ehr["glucose_level"] +
    0.3 * ehr["bmi"] +
    0.3 * ehr["cholesterol_level"]
)

ehr["cardiovascular_pathway_score"] = (
    0.5 * ehr["blood_pressure"] +
    0.25 * ehr["cholesterol_level"] +
    0.25 * ehr["heart_rate"]
)

ehr["respiratory_pathway_score"] = (
    0.6 * (100 - ehr["oxygen_saturation"]) +
    0.4 * ehr["heart_rate"]
)



In [15]:
# 4️⃣ Define feature groups
numerical_cols = [
    "age", "bmi", "blood_pressure", "cholesterol_level",
    "glucose_level", "heart_rate", "oxygen_saturation",
    "metabolic_pathway_score", "cardiovascular_pathway_score",
    "respiratory_pathway_score"
]

categorical_cols = ["gender", "medications"]

In [16]:
# 5️⃣ Encode target
label_encoder = LabelEncoder()
ehr["diagnosis"] = label_encoder.fit_transform(ehr["diagnosis"])
# ehr = ehr.drop(columns=['disease_risk'])

In [17]:
with open('ehr_dummy.csv','w') as f:
    ehr.to_csv(f,index=False)

In [18]:
X = ehr[['age', 'gender', 'bmi', 'blood_pressure', 'cholesterol_level', 
        'glucose_level', 'heart_rate', 'oxygen_saturation',
        'metabolic_pathway_score', 'cardiovascular_pathway_score',
        'respiratory_pathway_score', 'medications']]

y = ehr['diagnosis']

In [19]:
X.head()

,age,gender,bmi,blood_pressure,cholesterol_level,glucose_level,heart_rate,oxygen_saturation,metabolic_pathway_score,cardiovascular_pathway_score,respiratory_pathway_score,medications
0,69,Female,25.7,119.0,191.0,67.0,85,94.8,91.81,128.50,37.12,None
1,32,Female,22.3,131.0,253.0,109.0,80,96.8,126.19,148.75,33.92,Atorvastatin
2,89,Female,27.8,126.0,250.0,127.0,81,97.3,134.14,145.75,34.02,Insulin
3,78,Male,20.8,126.0,230.0,97.0,92,95.7,114.04,143.50,39.38,None
4,38,Male,27.2,150.0,219.0,59.0,92,96.1,97.46,152.75,39.14,Amlodipine


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [21]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [22]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [23]:
from sklearn.ensemble import RandomForestClassifier

In [24]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("Classifier", RandomForestClassifier(n_estimators=200, random_state=42))
])

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [26]:
model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('Classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [32]:
y_pred = model.predict(X_test)

In [28]:
from sklearn.metrics import roc_auc_score, classification_report

In [29]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       141
           1       1.00      1.00      1.00        86
           2       1.00      1.00      1.00       216
           3       1.00      1.00      1.00       433
           4       1.00      1.00      1.00       124

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [42]:
import shap

# ✅ Create SHAP explainer
explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test)

# ✅ Summary plot (global feature importance)
shap.summary_plot(shap_values, X_test, plot_type="bar")

# ✅ Detailed dependence plot (relationship for one feature)
shap.dependence_plot("age", shap_values.values, X_test)  # example: feature = "age"

# ✅ Individual patient explanation
shap.plots.waterfall(shap_values[0])  # shows how features contributed to one prediction


ModuleNotFoundError: No module named 'shap'

In [43]:
from lime import lime_tabular
import numpy as np

# ✅ Create LIME explainer
explainer = lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=X_train.columns,
    class_names=model.classes_,
    mode='classification'
)

# ✅ Choose one patient/sample
i = 5
exp = explainer.explain_instance(
    data_row=X_test.iloc[i],
    predict_fn=model.predict_proba
)

# ✅ Show explanation in notebook
exp.show_in_notebook(show_table=True)

# ✅ Or save as HTML
exp.save_to_file('lime_explanation.html')


ModuleNotFoundError: No module named 'lime'

In [41]:
import pickle
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)